# Context Sprawl: Doing the Work vs. Delegating It

Supporting data for section 9.1.1. Figure 9.2 makes the case for delegation
conceptually; this notebook measures it.

The task has two halves, and both are real computation:

- **A**: compute the Hailstone sequence for 27 and report how many steps it
  took.
- **B**: take that step count, double it, and compute the Hailstone sequence for
  that number too.

B's starting number comes out of A, so the second half genuinely depends on the
first one finishing rather than being two unrelated jobs glued together. Both
halves are the same kind of work, which makes the growth during A and the growth
during B directly comparable.

The task then runs twice:

1. **Single agent** performs both halves itself.
2. **Coordinator** delegates A to a subagent, receives only the step count back,
   and performs B itself.

Rollout size is sampled at four checkpoints in each run: start of A, end of A,
start of B, end of B. The delegated run adds a fifth, the subagent's own rollout
when it finishes A, which is what shows the work still happened somewhere.

27 is the smallest starting number whose sequence takes over 100 steps, and that
length is what makes the sprawl visible. A small number like 6 finishes in 8
steps and there is nothing to see.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("✓ Using Ollama Cloud")

## Setup

The same `next_number` tool Chapter 9's examples use. The `stop-at-one` skill in
`.agents/skills/` is discovered automatically and keeps the agent to one tool
call per step, so the step count tracks the sequence length. That discovery is
scoped to the working directory, so this notebook has to run from `examples/`.

A plain Python version of the sequence gives the expected lengths up front,
which fixes B's starting number and provides something to check the agents'
answers against.

In [ ]:
import json
import re
from pathlib import Path

from tokenizers import Tokenizer

from llm_agents_from_scratch import LLMAgent, LLMAgentBuilder
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.subagents import SubAgentSpec
from llm_agents_from_scratch.tools import SimpleFunctionTool

MAX_STEPS = 400
STEP_MARK = "=== Task Step Start ==="

tokenizer = Tokenizer.from_pretrained("Qwen/Qwen3-8B")


def n_tokens(text: str) -> int:
    """Count tokens in text using the Qwen tokenizer."""
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


model = "qwen3.5:397b-cloud" if use_cloud else "qwen3:14b"
host = "https://ollama.com" if use_cloud else None
llm = OllamaLLM(host=host, model=model, think=False, json_prompt_mode=use_cloud)


def next_number(x: int) -> int:
    """Apply one Hailstone step to x."""
    if x % 2 == 0:
        return x // 2
    return 3 * x + 1


next_number_tool = SimpleFunctionTool(func=next_number)


def hailstone_steps(x: int) -> int:
    """Count the steps in the Hailstone sequence starting at x."""
    steps = 0
    while x != 1:
        x = next_number(x)
        steps += 1
    return steps


A_START = 27
A_STEPS = hailstone_steps(A_START)
B_START = A_STEPS * 2
B_STEPS = hailstone_steps(B_START)

task_a = (
    f"First, compute the full Hailstone sequence for {A_START} step by step "
    "using next_number until you reach 1, and report how many steps it took."
)
task_b = (
    "Then take that step count, double it, and compute the full Hailstone "
    "sequence for that number the same way, reporting how many steps it took."
)

print(f"A: {A_START} takes {A_STEPS} steps")
print(f"B: {B_START} takes {B_STEPS} steps")

## Measuring the rollout

Rollout size is reported in tokens, counted with Qwen's own vocabulary through
the `tokenizers` package. That matters more than it sounds: this content is
dense with numbers and JSON punctuation, which tokenizes at roughly 2.8
characters per token rather than the usual rule-of-thumb 4, so an estimate
would have understated the counts by a third. Character counts are kept
alongside for reference.

The boundary between A and B is found after the run rather than by polling
mid-flight. The rollout is split on its step markers, and the first step whose
tool call carries B's starting number is where A ended and B began. Token counts
are then taken on the actual rollout prefix at each checkpoint.

The subagent's rollout needs one piece of instrumentation. `UseSubAgentTool`
builds the subagent, awaits its handler inline, and discards it, so nothing
holds a reference afterwards. Wrapping `LLMAgent.run` records every handler that
gets created, which makes the subagent's rollout reachable once the run is over.

In [ ]:
def split_steps(rollout: str) -> tuple[list[int], list[str]]:
    """Split a rollout into per-step parts and their cumulative offsets."""
    offsets, pos = [], rollout.find(STEP_MARK)
    while pos != -1:
        offsets.append(pos)
        pos = rollout.find(STEP_MARK, pos + 1)
    bounds = offsets + [len(rollout)]
    parts = [rollout[bounds[i] : bounds[i + 1]] for i in range(len(offsets))]
    return bounds, parts


def boundary_index(parts: list[str], start_number: int) -> int | None:
    """Find the step index whose tool call carries start_number."""
    for index, part in enumerate(parts):
        if re.search(rf'"x":\s*{start_number}\b', part):
            return index
    return None


def measure(handler, elapsed: float) -> dict:
    """Checkpoint a handler's rollout size at the A/B boundary."""
    rollout = handler.rollout
    bounds, parts = split_steps(rollout)
    split = boundary_index(parts, B_START)
    end_of_a = rollout[: bounds[split]] if split else ""
    return {
        "task_steps": handler.step_counter,
        "steps_in_A": split,
        "steps_in_B": len(parts) - split if split else None,
        "tokens": {
            "start_of_A": 0,
            "end_of_A": n_tokens(end_of_a),
            "start_of_B": n_tokens(end_of_a),
            "end_of_B": n_tokens(rollout),
        },
        "chars": {
            "start_of_A": 0,
            "end_of_A": len(end_of_a),
            "start_of_B": len(end_of_a),
            "end_of_B": len(rollout),
        },
        "seconds": round(elapsed, 1),
    }


captured_handlers = []
_llm_agent_run = LLMAgent.run


def _run_capturing(self, *args, **kwargs):
    handler = _llm_agent_run(self, *args, **kwargs)
    captured_handlers.append((self, handler))
    return handler


LLMAgent.run = _run_capturing

## Run 1: one agent does both halves

Every tool call and every result lands in this agent's own rollout, and the
rollout is what gets sent back to the LLM on the next step. B starts from
whatever A left behind.

In [ ]:
started = time.perf_counter()

solo = LLMAgent(llm=llm, tools=[next_number_tool])
solo_handler = solo.run(
    Task(instruction=f"{task_a} {task_b}"),
    max_steps=MAX_STEPS,
)
solo_result = await solo_handler

single_agent = measure(solo_handler, time.perf_counter() - started)
single_agent

## Run 2: the coordinator delegates A

The coordinator needs both the subagent and `next_number`, since it delegates
the first half and performs the second itself. The subagent still takes the same
number of steps. The difference is where those steps accumulate: in a rollout
that is discarded once the result comes back.

In [ ]:
started = time.perf_counter()
captured_handlers.clear()

spec = SubAgentSpec(
    name="hailstone",
    description="Computes Hailstone sequences using next_number.",
    builder=LLMAgentBuilder(llm=llm, tools=[next_number_tool]),
    max_steps=MAX_STEPS,
)
coordinator = LLMAgent(llm=llm, subagents=[spec], tools=[next_number_tool])
coord_handler = coordinator.run(
    Task(
        instruction=(
            "First, ask the hailstone subagent to compute the full Hailstone "
            f"sequence for {A_START} and report how many steps it took. "
            f"{task_b} Do that second computation yourself using next_number."
        ),
    ),
    max_steps=MAX_STEPS,
)
coord_result = await coord_handler

delegated = measure(coord_handler, time.perf_counter() - started)

subagent_handler = next(h for a, h in captured_handlers if a is not coordinator)
subagent_rollout = subagent_handler.rollout
delegated["subagent_end_of_A_tokens"] = n_tokens(subagent_rollout)
delegated["subagent_end_of_A_chars"] = len(subagent_rollout)
delegated["subagent_steps"] = subagent_handler.step_counter
delegated

## The checkpoints

The four shared checkpoints, plus the subagent's own view of the end of A. The
number to read is how much the rollout grew between the start of A and the start
of B: that is the cost of having done the work in-context rather than handing it
off.

In [ ]:
labels = [
    ("start of A", "start_of_A"),
    ("start of B (end of A)", "start_of_B"),
    ("end of B", "end_of_B"),
]

print(f"{'checkpoint':<24}{'single':>10}{'delegated':>12}   (tokens)")
for label, key in labels:
    print(
        f"{label:<24}{single_agent['tokens'][key]:>10,}"
        f"{delegated['tokens'][key]:>12,}",
    )
print(
    f"{'end of A (subagent)':<24}{'':>10}"
    f"{delegated['subagent_end_of_A_tokens']:>12,}",
)

print(f"\n{'checkpoint':<24}{'single':>10}{'delegated':>12}   (chars)")
for label, key in labels:
    print(
        f"{label:<24}{single_agent['chars'][key]:>10,}"
        f"{delegated['chars'][key]:>12,}",
    )
print(
    f"{'end of A (subagent)':<24}{'':>10}"
    f"{delegated['subagent_end_of_A_chars']:>12,}",
)

print(
    f"\n{'task steps':<24}{single_agent['task_steps']:>10}"
    f"{delegated['task_steps']:>12}",
)
print(
    f"{'seconds':<24}{single_agent['seconds']:>10}{delegated['seconds']:>12}",
)

solo_a = single_agent["tokens"]["start_of_B"]
coord_a = delegated["tokens"]["start_of_B"]
print(
    f"\nComputing A grew the single agent's rollout to {solo_a:,} tokens over "
    f"{single_agent['steps_in_A']} steps; the subagent's own rollout reached "
    f"{delegated['subagent_end_of_A_tokens']:,} tokens doing the same work, "
    f"while the coordinator's grew to only {coord_a:,} tokens over "
    f"{delegated['steps_in_A']} step(s).",
)

In [ ]:
data = {
    "a_start": A_START,
    "a_steps_expected": A_STEPS,
    "b_start": B_START,
    "b_steps_expected": B_STEPS,
    "model": model,
    "max_steps": MAX_STEPS,
    "single_agent": single_agent,
    "delegated": delegated,
    "single_agent_answer": solo_result.content,
    "delegated_answer": coord_result.content,
}
out_path = Path("ch09_context_sprawl_comparison.json")
out_path.write_text(json.dumps(data, indent=2))